In [1]:
!pip install datasets transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


Access the travel-related dataset directly from Hugging Face

In [33]:
from datasets import load_dataset, concatenate_datasets

# Load datasets
dataset1 = load_dataset("JasleenSingh91/travel-questions-response")
dataset2 = load_dataset("bitext/Bitext-travel-llm-chatbot-training-dataset")
dataset3 = load_dataset("nitv/fitness_chatbot_245")

# Combine all datasets
combined_dataset = concatenate_datasets([
    dataset1["train"],
    dataset2["train"],
    dataset3["train"],
])

# Inspect the combined dataset
print("Sample from combined dataset:", combined_dataset[0])

README.md:   0%|          | 0.00/9.21k [00:00<?, ?B/s]

(…)-travel-llm-chatbot-training-dataset.csv:   0%|          | 0.00/19.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/31658 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/345 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/104k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/245 [00:00<?, ? examples/s]

Sample from combined dataset: {'Input': 'Can you suggest the best travel destinations for my next trip?', 'Response_1': ' Absolutely! To help me suggest the best travel destinations for your next trip, I\'ll need to know a few things from you:\n1. What type of trip are you looking for? (e.g., beach vacation, adventure travel, city break, cultural tour, etc.)\n2. What is your preferred climate? (e.g., warm, cold, tropical, dry, etc.)\n3. How long will you be traveling for?\n4. What is your budget range?\n5. Do you have any specific travel preferences, such as accessibility for individuals with disabilities or dietary restrictions?\nBased on your answers to these questions, I can suggest some travel destinations that may fit your needs and preferences. Here are a few options to consider:\n1. Maui, Hawaii: If you\'re looking for a warm, tropical beach vacation, Maui is a great option. With beautiful beaches, fantastic weather, and a variety of activities, Maui has something for everyone.\

preprocessing

In [64]:
import random
import pandas as pd
from datasets import Dataset

def standardize_format(example):
    # Adjust based on column names in the datasets
    if "Input" in example and "Response_1" in example:
        return {"input": example["Input"], "output": example["Response_1"]}
    elif "questions" in example and "answers" in example:
        return {"input": example["questions"], "output": example["answers"]}
    elif "question" in example and "answer" in example:
        return {"input": example["question"], "output": example["answer"]}
    else:
        return None  # Ignore examples that don't match

# Apply standardization
standardized_dataset = combined_dataset.map(standardize_format, remove_columns=combined_dataset.column_names)

# Remove any None values
standardized_dataset = standardized_dataset.filter(lambda x: x is not None)

# Filter relevant examples based on keywords
def filter_relevant_examples(example):
    # Check if the 'input' field exists and is not None
    if example.get("input") and isinstance(example["input"], str):
        keywords = ["route", "run", "trail", "path", "distance", "terrain", "location", "fitness", "park"]
        return any(keyword in example["input"].lower() for keyword in keywords)
    return False  # Exclude examples without a valid 'input' field

filtered_dataset = standardized_dataset.filter(filter_relevant_examples)

# Adapt responses to the running domain (NEED TO CHANGE !!!)
def adapt_responses(example):
    if "travel" in example["input"].lower():
        example["output"] = "I can suggest running routes in that location. Tell me your preferred distance or terrain!"
    elif "hotel" in example["output"].lower():
        example["output"] = example["output"].replace("hotel", "running route")
    elif "place" in example["input"].lower():
        example["output"] = "This location has great running paths. Let me find a route for you!"
    return example

adapted_dataset = filtered_dataset.map(adapt_responses)


# Generate synthetic running route data
def generate_synthetic_data(num_examples=300):
    locations = ["work", "current location", "home"]
    distances = ["5 km", "10 km", "3 km", "7 km"]
    terrains = ["trail", "flat", "scenic", "urban"]

    synthetic_data = []
    for _ in range(num_examples):
        location = random.choice(locations)
        distance = random.choice(distances)
        terrain = random.choice(terrains)
        input_text = f"Suggest a {distance} {terrain} route near {location}."
        output_text = f"A {distance} {terrain} route near {location}."
        synthetic_data.append({"input": input_text, "output": output_text})

    return synthetic_data

synthetic_dataset = generate_synthetic_data()
synthetic_df = pd.DataFrame(synthetic_dataset)

# Combine adapted and synthetic datasets
adapted_df = pd.DataFrame(adapted_dataset)
augmented_dataset = pd.concat([adapted_df, synthetic_df]).reset_index(drop=True)

# Convert to Hugging Face Dataset format
final_dataset = Dataset.from_pandas(augmented_dataset)

# Save the final dataset for reuse
final_dataset.save_to_disk("final_running_chatbot_dataset")

# Verify the dataset
print(f"Final dataset size: {len(final_dataset)}")
print(f"Sample entry: {final_dataset[0]}")


Saving the dataset (0/1 shards):   0%|          | 0/1219 [00:00<?, ? examples/s]

Final dataset size: 1219
Sample entry: {'output': " Absolutely! If you're looking for an exciting and off-the-beaten-path destination for your next vacation, I would highly recommend the beautiful and adventurous country of Bhutan. Nestled in the Eastern Himalayas, Bhutan is known for its breathtaking landscapes, rich cultural heritage, and commitment to sustainability and environmental preservation. \n\nOne of the most unique aspects of Bhutan is its policy of high value, low impact tourism. This means that the country limits the number of tourists that enter each year, ensuring that visitors have a truly authentic and memorable experience. \n\nSome must-see attractions in Bhutan include the stunning Tiger's Nest Monastery, perched on a cliffside overlooking the Paro Valley; the colorful and bustling markets of Thimphu, the capital city; and the peaceful Phobjikha Valley, home to the endangered black-necked cranes.\n\nFor the adventurous traveler, Bhutan offers a range of activities i

model select

In [65]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"  # A lightweight GPT-2 variant
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


Format the dataset for training

In [66]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenize Dataset
def preprocess_function(example):
    # Combine input and output into a single sequence for causal language modeling
    text = f"Question: {example['input']} Answer: {example['output']}"

    # Tokenize the combined text
    tokenized = tokenizer(
        text,
        padding="max_length",    # Pad to a fixed length for uniformity
        truncation=True,         # Truncate sequences longer than max_length
    )

    # Set input_ids as labels for causal language modeling
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_datasets = final_dataset.map(preprocess_function)





Map:   0%|          | 0/1219 [00:00<?, ? examples/s]

Fine-Tune the Model

In [67]:
from transformers import Trainer, TrainingArguments
import os
os.environ["WANDB_DISABLED"] = "true"

# Split Dataset
split_dataset = tokenized_datasets.train_test_split(test_size=0.2)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    save_total_limit=1,
    report_to="none",
    logging_dir="./logs",
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [68]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.125800,0.580162
2,0.571100,0.543922
3,0.551900,0.527962
4,0.533100,0.520663
5,0.513900,0.516159
6,0.514000,0.514654


TrainOutput(global_step=732, training_loss=0.6120684172937779, metrics={'train_runtime': 1446.5106, 'train_samples_per_second': 4.044, 'train_steps_per_second': 0.506, 'total_flos': 1528585990963200.0, 'train_loss': 0.6120684172937779, 'epoch': 6.0})

Generate Responses

In [70]:
# Generate responses using the fine-tuned model
def generate_response(input_text, max_length=50):
    # Encode the input text
    input_ids = tokenizer.encode(input_text, return_tensors="pt")

    # Move input_ids to the same device as the model
    input_ids = input_ids.to(model.device)

    # Generate a response
    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True,  # Enable sampling for temperature to take effect
        pad_token_id=tokenizer.eos_token_id,  # Avoid warnings about padding
    )

    # Decode and return the response
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Test the model
input_text = "Plan a route for me from my home to work."
response = generate_response(input_text)
print("Generated Response:", response)


Generated Response: Plan a route for me from my home to work. Here, I'd like to suggest a scenic route for you: the Appalachian Trail, which runs from Annapolis to the Virginia Tech USA, is a great option for me. I'd be happy


Save and Reload the Model

In [26]:
# Save the model
model.save_pretrained("./my_travel_bot")
tokenizer.save_pretrained("./my_travel_bot")

# Reload the model
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("./my_travel_bot")
tokenizer = AutoTokenizer.from_pretrained("./my_travel_bot")
